# Celeb-DF++ Stage 1 Notebook

## Goal
This notebook is for the first stage of the bachelor thesis workflow:
- inspect the dataset structure,
- build a video manifest,
- extract basic video metadata,
- run practical EDA,
- create a small curated pilot subset,
- save reusable metadata tables for the next stages.

## Scope
This notebook does **not** train models. It is only for dataset overview, subset selection, metadata building, and light preprocessing preparation.

## Notebook Structure
1. Imports and config
2. Dataset structure inspection
3. Dataset scanning and manifest creation
4. Video metadata extraction
5. Basic EDA
6. Curated subset creation
7. Split assignment and next-stage preparation (to add next)

In [ ]:
from pathlib import Path
import re
import hashlib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

# ---------------------------
# CONFIG
# ---------------------------
DATASET_ROOT = Path("/home/n-salikhova/datasets-retry/extracted/Celeb-DF-v3")
OUTPUT_DIR = Path("../metadata").resolve()
RANDOM_SEED = 42
PILOT_REAL_COUNT = 100
PILOT_FAKE_COUNT = 100
MIN_DURATION_SEC = 3.0
MIN_RESOLUTION = (320, 240)  # (width, height)
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
MAX_VIDEOS_FOR_METADATA = None  # set to int for a quick dry run
SAVE_PARQUET = True

FULL_MANIFEST_CSV = OUTPUT_DIR / "full_manifest.csv"
CLEAN_MANIFEST_CSV = OUTPUT_DIR / "clean_manifest.csv"
PILOT_MANIFEST_CSV = OUTPUT_DIR / "pilot_subset_manifest.csv"

FULL_MANIFEST_PARQUET = OUTPUT_DIR / "full_manifest.parquet"
CLEAN_MANIFEST_PARQUET = OUTPUT_DIR / "clean_manifest.parquet"
PILOT_MANIFEST_PARQUET = OUTPUT_DIR / "pilot_subset_manifest.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"Exists:       {DATASET_ROOT.exists()}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")

## 1. Dataset Structure Inspection

Start with a lightweight directory inspection before building any manifests. This helps confirm the dataset root, major subsets, and whether the structure matches expectations.

In [ ]:
top_entries = sorted(DATASET_ROOT.iterdir()) if DATASET_ROOT.exists() else []

print(f"Top-level entries under dataset root: {len(top_entries)}")
for entry in top_entries[:20]:
    kind = "dir" if entry.is_dir() else "file"
    print(f"- [{kind}] {entry.name}")

video_files = [
    path for path in DATASET_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
]

print(f"\nDiscovered video files: {len(video_files):,}")

if video_files:
    sample_paths = pd.DataFrame({
        "sample_relative_path": [str(path.relative_to(DATASET_ROOT)) for path in video_files[:10]]
    })
    display(sample_paths)

## 2. Dataset Scanning and Manifest Creation

Scan all available video files and build a first manifest. The code tries to infer label, split, source subset, manipulation type, and identity from the directory structure and filenames when possible.

In [ ]:
REAL_HINTS = {"real", "celeb-real", "youtube-real", "authentic"}
FAKE_HINTS = {"fake", "deepfake", "synthesis", "manipulated"}
SPLIT_MAP = {
    "train": "train",
    "val": "val",
    "valid": "val",
    "validation": "val",
    "dev": "val",
    "test": "test",
}


def make_video_id(relative_path: Path) -> str:
    stem = relative_path.with_suffix("").as_posix()
    digest = hashlib.md5(relative_path.as_posix().encode("utf-8")).hexdigest()[:8]
    return f"{stem.replace('/', '__')}__{digest}"


def infer_split(relative_path: Path) -> str | None:
    for part in relative_path.parts:
        key = part.lower()
        if key in SPLIT_MAP:
            return SPLIT_MAP[key]
    return None


def infer_label(relative_path: Path) -> str:
    parts_lower = [part.lower() for part in relative_path.parts]
    if any(part in {"celeb-real", "youtube-real", "real"} for part in parts_lower):
        return "real"
    if "celeb-synthesis" in parts_lower:
        return "fake"
    if any(part in FAKE_HINTS for part in parts_lower):
        return "fake"
    if any(part in REAL_HINTS for part in parts_lower):
        return "real"
    return "unknown"


def infer_source_subset(relative_path: Path) -> str | None:
    return relative_path.parts[0] if relative_path.parts else None


def infer_manipulation_family(relative_path: Path, label: str) -> str | None:
    parts = relative_path.parts
    if label == "fake" and len(parts) >= 2:
        if parts[0].lower() == "celeb-synthesis":
            return parts[1]
    return None


def infer_manipulation_type(relative_path: Path, label: str) -> str | None:
    parts = relative_path.parts
    if label == "fake" and len(parts) >= 3:
        if parts[0].lower() == "celeb-synthesis":
            return parts[2]
        return parts[1]
    if label == "real" and parts:
        return parts[0]
    return None


def infer_identity_info(path: Path) -> tuple[str | None, str | None, str | None]:
    ids = re.findall(r"id\d+", path.stem)
    if not ids:
        return None, None, None
    identity = ids[0]
    source_identity = ids[0]
    target_identity = ids[1] if len(ids) > 1 else None
    return identity, source_identity, target_identity


manifest_rows = []
for video_path in tqdm(video_files, desc="Scanning videos"):
    relative_path = video_path.relative_to(DATASET_ROOT)
    label = infer_label(relative_path)
    identity, source_identity, target_identity = infer_identity_info(video_path)
    manifest_rows.append({
        "video_id": make_video_id(relative_path),
        "path": str(video_path.resolve()),
        "relative_path": str(relative_path),
        "split": infer_split(relative_path),
        "label": label,
        "source_subset": infer_source_subset(relative_path),
        "manipulation_family": infer_manipulation_family(relative_path, label),
        "manipulation_type": infer_manipulation_type(relative_path, label),
        "identity": identity,
        "source_identity": source_identity,
        "target_identity": target_identity,
        "file_ext": video_path.suffix.lower(),
    })

manifest_df = pd.DataFrame(manifest_rows)
print(f"Manifest rows: {len(manifest_df):,}")
display(manifest_df.head())

print("\nLabel counts:")
display(manifest_df["label"].value_counts(dropna=False).rename_axis("label").to_frame("count"))

## 3. Video Metadata Extraction

Now enrich the manifest with practical video-level metadata: readable stream status, duration, FPS, frame count, width, height, and file size. These fields are needed both for EDA and for filtering unusable videos before subset construction.

In [ ]:
def extract_video_metadata(video_path: Path) -> dict:
    result = {
        "readable": False,
        "error": None,
        "frame_count": np.nan,
        "fps": np.nan,
        "duration_sec": np.nan,
        "width": np.nan,
        "height": np.nan,
        "file_size_bytes": np.nan,
        "file_size_mb": np.nan,
    }

    try:
        result["file_size_bytes"] = video_path.stat().st_size
        result["file_size_mb"] = result["file_size_bytes"] / (1024 ** 2)
    except Exception as exc:
        result["error"] = f"stat_error: {exc}"
        return result

    capture = cv2.VideoCapture(str(video_path))
    try:
        if not capture.isOpened():
            result["error"] = "cannot_open"
            return result

        frame_count = capture.get(cv2.CAP_PROP_FRAME_COUNT)
        fps = capture.get(cv2.CAP_PROP_FPS)
        width = capture.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = capture.get(cv2.CAP_PROP_FRAME_HEIGHT)
        ok, _ = capture.read()

        if not ok:
            result["error"] = "cannot_read_first_frame"
            return result

        result["readable"] = True
        result["frame_count"] = int(frame_count) if frame_count and frame_count > 0 else np.nan
        result["fps"] = float(fps) if fps and fps > 0 else np.nan
        result["width"] = int(width) if width and width > 0 else np.nan
        result["height"] = int(height) if height and height > 0 else np.nan

        if pd.notna(result["frame_count"]) and pd.notna(result["fps"]) and result["fps"] > 0:
            result["duration_sec"] = result["frame_count"] / result["fps"]
    except Exception as exc:
        result["error"] = f"capture_error: {exc}"
    finally:
        capture.release()

    return result


metadata_input_df = manifest_df.copy()
if MAX_VIDEOS_FOR_METADATA is not None:
    metadata_input_df = metadata_input_df.head(MAX_VIDEOS_FOR_METADATA).copy()

metadata_rows = []
for row in tqdm(metadata_input_df.itertuples(index=False), total=len(metadata_input_df), desc="Extracting metadata"):
    metadata = extract_video_metadata(Path(row.path))
    metadata_rows.append({"video_id": row.video_id, **metadata})

metadata_df = pd.DataFrame(metadata_rows)
full_df = manifest_df.merge(metadata_df, on="video_id", how="left")

full_df.to_csv(FULL_MANIFEST_CSV, index=False)
if SAVE_PARQUET:
    full_df.to_parquet(FULL_MANIFEST_PARQUET, index=False)

print(f"Saved full manifest: {FULL_MANIFEST_CSV}")
print(f"Readable videos: {int(full_df['readable'].fillna(False).sum()):,} / {len(full_df):,}")
display(full_df.head())

## 4. Basic EDA

Use the extracted metadata to inspect class balance, manipulation coverage, and basic technical properties of the videos. This is enough for a clean first-stage understanding of the benchmark.

In [ ]:
eda_df = full_df[full_df["readable"] == True].copy()
eda_df["resolution"] = eda_df["width"].astype("Int64").astype(str) + "x" + eda_df["height"].astype("Int64").astype(str)

print("EDA input rows:", len(eda_df))
print("Unreadable / corrupted rows:", int((full_df["readable"] != True).sum()))

summary_table = pd.DataFrame({
    "metric": [
        "total_videos",
        "readable_videos",
        "real_videos",
        "fake_videos",
        "unique_manipulation_types",
        "median_duration_sec",
        "median_fps",
        "median_file_size_mb",
    ],
    "value": [
        len(full_df),
        len(eda_df),
        int((eda_df["label"] == "real").sum()),
        int((eda_df["label"] == "fake").sum()),
        int(eda_df["manipulation_type"].nunique(dropna=True)),
        round(float(eda_df["duration_sec"].median()), 2) if not eda_df.empty else np.nan,
        round(float(eda_df["fps"].median()), 2) if not eda_df.empty else np.nan,
        round(float(eda_df["file_size_mb"].median()), 2) if not eda_df.empty else np.nan,
    ],
})
display(summary_table)

display(
    eda_df[[
        "video_id", "label", "source_subset", "manipulation_family", "manipulation_type",
        "duration_sec", "fps", "width", "height", "file_size_mb"
    ]].sample(min(8, len(eda_df)), random_state=RANDOM_SEED)
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sns.countplot(data=eda_df, x="label", order=eda_df["label"].value_counts().index, ax=axes[0, 0])
axes[0, 0].set_title("Label Distribution")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Count")

manip_counts = (
    eda_df[eda_df["label"] == "fake"]["manipulation_type"]
    .fillna("unknown")
    .value_counts()
    .head(15)
    .sort_values()
)
manip_counts.plot(kind="barh", ax=axes[0, 1], color="tomato")
axes[0, 1].set_title("Manipulation Type Counts")
axes[0, 1].set_xlabel("Count")

sns.histplot(data=eda_df, x="duration_sec", bins=40, ax=axes[0, 2])
axes[0, 2].set_title("Duration Distribution")
axes[0, 2].set_xlabel("Duration (sec)")

sns.histplot(data=eda_df, x="fps", bins=30, ax=axes[1, 0])
axes[1, 0].set_title("FPS Distribution")
axes[1, 0].set_xlabel("FPS")

sns.scatterplot(data=eda_df.sample(min(3000, len(eda_df)), random_state=RANDOM_SEED), x="width", y="height", hue="label", alpha=0.6, ax=axes[1, 1])
axes[1, 1].set_title("Resolution Distribution")
axes[1, 1].set_xlabel("Width")
axes[1, 1].set_ylabel("Height")

sns.histplot(data=eda_df, x="file_size_mb", bins=40, ax=axes[1, 2])
axes[1, 2].set_title("File Size Distribution")
axes[1, 2].set_xlabel("File size (MB)")

plt.tight_layout()
plt.show()

n_real = int((eda_df["label"] == "real").sum())
n_fake = int((eda_df["label"] == "fake").sum())
if n_real > 0:
    print(f"Class imbalance: 1 real : {n_fake / n_real:.2f} fake")

## 5. Curated Subset Creation

Apply simple and practical cleaning rules before building a pilot subset:
- keep only readable videos,
- remove corrupted files,
- exclude extremely short videos,
- optionally enforce a minimum resolution,
- sample a balanced pilot subset of real and fake videos.

In [ ]:
clean_mask = (
    (full_df["readable"] == True)
    & (full_df["duration_sec"].fillna(0) >= MIN_DURATION_SEC)
    & (full_df["width"].fillna(0) >= MIN_RESOLUTION[0])
    & (full_df["height"].fillna(0) >= MIN_RESOLUTION[1])
)

clean_df = full_df.loc[clean_mask].copy()
clean_df["quality_flag"] = "keep"

excluded_df = full_df.loc[~clean_mask].copy()
print(f"Clean manifest rows: {len(clean_df):,}")
print(f"Excluded rows:       {len(excluded_df):,}")

clean_df.to_csv(CLEAN_MANIFEST_CSV, index=False)
if SAVE_PARQUET:
    clean_df.to_parquet(CLEAN_MANIFEST_PARQUET, index=False)


def balanced_fake_sample(fake_df: pd.DataFrame, target_n: int, seed: int) -> pd.DataFrame:
    if fake_df.empty or target_n <= 0:
        return fake_df.head(0).copy()

    if fake_df["manipulation_type"].notna().sum() == 0:
        return fake_df.sample(min(target_n, len(fake_df)), random_state=seed).copy()

    grouped = list(fake_df.groupby("manipulation_type", dropna=False, sort=True))
    base_take = target_n // len(grouped)
    remainder = target_n % len(grouped)

    sampled_parts = []
    used_ids = set()
    for index, (_, group_df) in enumerate(grouped):
        take_n = min(len(group_df), base_take + (1 if index < remainder else 0))
        if take_n <= 0:
            continue
        part = group_df.sample(take_n, random_state=seed + index).copy()
        sampled_parts.append(part)
        used_ids.update(part["video_id"].tolist())

    sampled_df = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else fake_df.head(0).copy()

    if len(sampled_df) < min(target_n, len(fake_df)):
        remaining = fake_df.loc[~fake_df["video_id"].isin(used_ids)]
        extra_n = min(target_n - len(sampled_df), len(remaining))
        if extra_n > 0:
            extra = remaining.sample(extra_n, random_state=seed + 999).copy()
            sampled_df = pd.concat([sampled_df, extra], ignore_index=True)

    return sampled_df.head(min(target_n, len(fake_df))).copy()


real_candidates = clean_df[clean_df["label"] == "real"].copy()
fake_candidates = clean_df[clean_df["label"] == "fake"].copy()

pilot_real_df = real_candidates.sample(min(PILOT_REAL_COUNT, len(real_candidates)), random_state=RANDOM_SEED).copy()
pilot_fake_df = balanced_fake_sample(fake_candidates, target_n=PILOT_FAKE_COUNT, seed=RANDOM_SEED)

pilot_subset_df = pd.concat([pilot_real_df, pilot_fake_df], ignore_index=True)
pilot_subset_df = pilot_subset_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
pilot_subset_df["subset_name"] = "pilot"

pilot_subset_df.to_csv(PILOT_MANIFEST_CSV, index=False)
if SAVE_PARQUET:
    pilot_subset_df.to_parquet(PILOT_MANIFEST_PARQUET, index=False)

print(f"Saved clean manifest:  {CLEAN_MANIFEST_CSV}")
print(f"Saved pilot manifest:  {PILOT_MANIFEST_CSV}")
print(f"Pilot subset size:     {len(pilot_subset_df):,}")
print(f"Pilot real videos:     {(pilot_subset_df['label'] == 'real').sum():,}")
print(f"Pilot fake videos:     {(pilot_subset_df['label'] == 'fake').sum():,}")

display(
    pilot_subset_df.groupby(["label", "manipulation_family", "manipulation_type"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["label", "manipulation_family", "manipulation_type"])
)

## 6. Next Sections to Add

The next practical additions should be:
1. split assignment for the pilot subset (`train` / `val` / `test`), preferably identity-disjoint when identity metadata is reliable;
2. export of the final split-aware pilot manifest;
3. frame extraction preparation;
4. face extraction preparation;
5. emotion annotation preparation;
6. detector inference preparation.

At this point the notebook already produces:
- full manifest,
- clean manifest,
- pilot subset manifest,
- practical EDA plots,
- reusable video-level metadata.